In [ ]:
import sys
from pathlib import Path

import cooler

sys.path.insert(1, "..")
import bioframe as bf
import numpy as np
import pandas as pd
import pyranges as pr
from Bio import SeqIO

from config.eda import DataConfig
from config.model_conf import FeaturesConfig
from RNADNA_background.features_preparation import (
    GC_content_count,
    restriction_site_count,
)

In [ ]:
# configuration of the dataset
data_conf_file = Path("../config/data_conf.ini")
params = DataConfig(data_conf_file)

# configuration of the features
features_conf_file = Path("../config/features_conf.ini")
features_params = FeaturesConfig(features_conf_file, params.chromosomes)

RESTRICTASE = "AluI"
sites = {
    "AluI": "AGCT",  # grid-seq
}

# cool file to get bins coordinates
cool_file = params.hic_folder / Path(
    f"{params.cell_line}.mcool::resolutions/{params.bin_size}"
)
c = cooler.Cooler(str(cool_file))

## Windows creation

In [ ]:
# compartments annotation obtained from 4DNucleome
# converted from bigwig to bg
compartments = pd.read_csv(
    params.annotation_folder / f"{params.cell_line}_compartments.bg",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "Vec"],
)

compartments = compartments[~compartments["Vec"].isna()]
compartments["comp"] = "A"
compartments.loc[compartments["Vec"] < 0, "comp"] = "B"
compartments = compartments.drop("Vec", axis=1)
compartments_merged = bf.merge(compartments, min_dist=0, on=["comp"]).drop(
    "n_intervals", axis=1
)

compartments_merged.to_csv(
    params.annotation_folder / f"{params.cell_line}_compartments.bed",
    header=False,
    index=False,
    sep="\t",
)

In [ ]:
# creating windows from loci, annoteded by A/B compartment

compartments = pd.read_csv(
    params.annotation_folder / f"{params.cell_line}_compartments.bed",
    sep="\t",
    names=["chrom", "start", "end", "compartment"],
)

merged_compartments = bf.merge(
    compartments[["chrom", "start", "end"]], min_dist=0
).drop("n_intervals", axis=1)

merged_compartments = merged_compartments[
    merged_compartments.end - merged_compartments.start
    >= features_params.window_size
]

windows = []
for i, (chrom, start, end) in merged_compartments.iterrows():
    if chrom == "chrY":
        break
    for window_start in range(start, end, features_params.shift):
        window_end = window_start + features_params.window_size
        # current window is beyond interval end, but less than shift, then take a window from the end
        if window_end > end and window_end < end + features_params.shift:
            windows.append(
                {
                    "chr": chrom,
                    "start": end - features_params.window_size,
                    "end": end,
                }
            )
            break
        elif window_end < end:
            windows.append(
                {"chr": chrom, "start": window_start, "end": window_end}
            )

windows = pd.DataFrame(windows)
assert np.sum(windows.duplicated()) == 0

In [ ]:
windows.to_csv(
    params.learn_data_folder
    / f"windows_{features_params.window_size}_{features_params.shift}.tsv",
    sep="\t",
    index=False,
)

## Features generation

In [ ]:
# bins from cool files of hi-c data
bins = c.bins()[:][["chrom", "start", "end"]]
bins["bin"] = bins["start"] // params.bin_size
bins = bins[bins["chrom"] != "chrY"]

### Encode peaks & genes & repeats

In [ ]:
# encode peaks loading
data_types = [
    "ATAC-seq",
    "DNase-seq",
    "CTCF",
    "H3K4me3",
    "H3K27me3",
    "H3K27ac",
    "H3K4me1",
    "H3K9ac",
    "H3K9me3",
    "H3K36me3",
    "DRIP",
]
names = ["chrom", "start", "end"]
datasets = {
    data_type: pd.read_csv(
        params.annotation_folder
        / f"encode_peaks/{params.cell_line}_{data_type}.bed",
        sep="\t",
        header=None,
        names=names,
        usecols=[0, 1, 2],
    )
    for data_type in data_types
}

In [ ]:
# protein coding genes coverage
genes = pr.read_gtf(params.genome_folder / "gencode.vM21.annotation.gtf").df
genes = genes[["Chromosome", "Start", "End", "gene_type"]].rename(
    {"Chromosome": "chrom", "Start": "start", "End": "end"}, axis=1
)
genes = (
    genes[genes["gene_type"] == "protein_coding"]
    .drop("gene_type", axis=1)
    .reset_index(drop=True)
)
genes = bf.merge(genes, min_dist=0).drop("n_intervals", axis=1)

In [ ]:
# repeats coverage
repeats = pd.read_csv(
    params.genome_folder / f"repeats_{params.genome}.bed",
    sep="\t",
    header=None,
    usecols=[0, 1, 2],
    names=["chrom", "start", "end"],
)
repeats = bf.merge(repeats, min_dist=0).drop("n_intervals", axis=1)

In [ ]:
# triplexes coverage
triplexes = pd.read_csv(params.annotation_folder / "triplexes.tsv", sep="\t")
triplexes = triplexes.rename(
    {"Duplex-ID": "chrom", "Start": "start", "End": "end"}, axis=1
)[["chrom", "start", "end"]]
triplexes = triplexes[
    triplexes["chrom"]
    .str.slice(3, 5)
    .isin([str(i) for i in params.chromosomes])
]
triplexes = bf.merge(triplexes, min_dist=0).drop("n_intervals", axis=1)

In [ ]:
# data tracks binning

counts = bins.copy()
for data_type in data_types:
    counts = bf.count_overlaps(counts, datasets[data_type])
    counts = counts.rename({"count": f"count_{data_type}"}, axis=1)

counts = counts[counts["chrom"] != "chrY"]

counts = bf.coverage(counts, repeats)
counts = counts.rename({"coverage": "repeats"}, axis=1)

counts = bf.coverage(counts, genes)
counts = counts.rename({"coverage": "genes"}, axis=1)

counts = bf.coverage(counts, triplexes)
counts = counts.rename({"coverage": "triplexes"}, axis=1)

### Sequence features

In [ ]:
genome_fasta = SeqIO.to_dict(
    SeqIO.parse(
        params.genome_path,
        "fasta",
    )
)

In [ ]:
rest_per_bin = restriction_site_count(
    bins.rename({"chrom": "dna_chr"}, axis=1),
    genome_fasta,
    sites[RESTRICTASE],
    params,
)
rest_gc_per_bin = GC_content_count(rest_per_bin, genome_fasta, params)
rest_gc_per_bin = rest_gc_per_bin.rename({"dna_chr": "chrom"}, axis=1)

In [ ]:
features = counts.merge(
    rest_gc_per_bin[["chrom", "bin", "restriction_sites", "gc_count"]],
    how="left",
    on=["chrom", "bin"],
)

### Hi-C features

In [ ]:
# A/B compartments

compartments = pd.read_csv(
    params.annotation_folder / f"{params.cell_line}_compartments.bed",
    sep="\t",
    names=["chrom", "start", "end", "compartment"],
)

features = (
    bf.overlap(features, compartments, how="left")
    .drop(["chrom_", "start_", "end_"], axis=1)
    .rename({"compartment_": "compartment"}, axis=1)
)

features.loc[features["compartment"] == "A", "compartment"] = 1
features.loc[features["compartment"] == "B", "compartment"] = 0
features["compartment"] = features["compartment"].fillna(0)

# 1st Hi-C matrix eigenvector
compartments_bg = pd.read_csv(
    params.annotation_folder / f"{params.cell_line}_compartments.bg",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "compartment_vec"],
)
compartments_bg["compartment_vec"] = compartments_bg["compartment_vec"].fillna(
    0
)

features = (
    bf.overlap(
        features,
        compartments_bg,
        how="left",
    )
    .drop(["chrom_", "start_", "end_"], axis=1)
    .rename({"compartment_vec_": "compartment_vec"}, axis=1)
)

In [ ]:
# tads borders and mean contacts freq along the main diagonal

tads = []
mean_cf = []
for chromosome in params.chromosomes:
    tads_chr = pd.read_csv(
        params.annotation_folder / f"tads/tads_chr{chromosome}.bed",
        sep="\t",
        names=["chrom", "start", "end", "region"],
        dtype={"start": int, "end": int},
    )
    mean_cf_chr = pd.read_csv(
        params.annotation_folder / f"tads/tads_chr{chromosome}.binSignal",
        sep="\t",
        usecols=["chr", "from.coord", "to.coord", "mean.cf"],
        dtype={"from.coord": int, "to.coord": int},
    )
    tads.append(tads_chr)
    mean_cf.append(mean_cf_chr)
tads = pd.concat(tads)
mean_cf = pd.concat(mean_cf).rename(
    {"chr": "chrom", "from.coord": "start", "to.coord": "end"}, axis=1
)
tads.loc[tads["region"] == "domain", "tad"] = 1
tads.loc[tads["region"] != "domain", "tad"] = 0
tads = tads.drop("region", axis=1)

# chromatin loops
loops = pd.read_csv(
    params.annotation_folder
    / f"{params.cell_line}_loops_clustered_coords.tsv",
    sep="\t",
)
loops = bf.merge(loops, min_dist=0).drop("n_intervals", axis=1)
loops["loop"] = 1

In [ ]:
features = (
    bf.overlap(
        features,
        tads,
        how="left",
    )
    .drop(["chrom_", "start_", "end_"], axis=1)
    .rename({"tad_": "tad"}, axis=1)
)

features = (
    bf.overlap(
        features,
        mean_cf,
        how="left",
    )
    .drop(["chrom_", "start_", "end_"], axis=1)
    .rename({"mean.cf_": "mean_cf"}, axis=1)
)
features["mean_cf"] = features["mean_cf"].fillna(0)

features = (
    bf.overlap(
        features,
        loops,
        how="left",
    )
    .drop(["chrom_", "start_", "end_"], axis=1)
    .rename({"loop_": "loop"}, axis=1)
)
features["loop"] = features["loop"].fillna(0)

In [ ]:
# rna-seq + and - strands loading

rna_seq_plus = pd.read_csv(
    params.annotation_folder / f"{params.cell_line}_RNAseq_plus.bg",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "signal"],
)
rna_seq_plus["bin"] = (
    (rna_seq_plus["start"] + rna_seq_plus["end"]) // 2
) // params.bin_size
rna_seq_plus = (
    rna_seq_plus.groupby(["chrom", "bin"])["signal"].sum().reset_index()
)

rna_seq_minus = pd.read_csv(
    params.annotation_folder / f"{params.cell_line}_RNAseq_minus.bg",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "signal"],
)
rna_seq_minus["bin"] = (
    (rna_seq_minus["start"] + rna_seq_minus["end"]) // 2
) // params.bin_size
rna_seq_minus = (
    rna_seq_minus.groupby(["chrom", "bin"])["signal"].sum().reset_index()
)

In [ ]:
# averaging - and + strand signal
rna_seq_counts = bins[["chrom", "bin"]].copy()
rna_seq_counts = rna_seq_counts.merge(
    rna_seq_plus, how="left", on=["chrom", "bin"], suffixes=["", "_plus"]
)
rna_seq_counts = rna_seq_counts.merge(
    rna_seq_minus,
    how="left",
    on=["chrom", "bin"],
    suffixes=["_plus", "_minus"],
)
rna_seq_counts = rna_seq_counts.fillna(0)
rna_seq_counts["RNAseq_signal"] = (
    rna_seq_counts["signal_plus"] + rna_seq_counts["signal_minus"]
) / 2

In [ ]:
features = features.merge(
    rna_seq_counts[["chrom", "bin", "RNAseq_signal"]],
    on=["chrom", "bin"],
    how="left",
)

### DNase-seq coverage

In [ ]:
dnase_cov = pd.read_csv(
    params.annotation_folder / f"{params.cell_line}_DNase-seq_cov.bg",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "signal"],
)
dnase_cov["bin"] = (
    (dnase_cov["start"] + dnase_cov["end"]) // 2
) // params.bin_size
dnase_cov = dnase_cov.groupby(["chrom", "bin"])["signal"].sum().reset_index()
dnase_cov = dnase_cov.rename({"signal": "DNase-seq_cov"}, axis=1)
features = features.merge(
    dnase_cov[["chrom", "bin", "DNase-seq_cov"]],
    on=["chrom", "bin"],
    how="left",
)
features["DNase-seq_cov"] = features["DNase-seq_cov"].fillna(0)

In [ ]:
features.to_csv(
    params.learn_data_folder / f"features_{params.bin_size}.tsv",
    sep="\t",
    index=False,
)